# MIL-CREDA frente a CREDA — fase uno: la escalera

Este cuaderno no corre nada. Lee lo que dejó `Benchmark_Phase1_Run.ipynb` bajo `MIL-CREDA/Results/Benchmark/` — `summary.json`, `runs.jsonl` y el registro de la búsqueda del techo — y arma las tablas, las figuras y el informe a partir de esos archivos. Corregir una frase del informe cuesta volver a correr esta celda en adelante, segundos, y no la campaña entera.
Diez métodos sobre seis transferencias. Cada peldaño se diferencia del anterior en
**una sola cosa**, así una diferencia se puede atribuir a esa cosa y a ninguna otra.

El nombre dice qué le falta al método: un asterisco marca un componente ausente,
dos marcan dos, y un nombre sin marca es el método completo.

| id | nombre | adaptación | ponderación | término local | instancias que usa |
|---|---|---|---|---|---|
| `A` | `Baseline` | — | — | — | todas |
| `C` | `CREDA*` | CREDA | — | — | todas |
| `D` | `CREDA` | CREDA | sí | — | todas |
| `B` | `MIL-Baseline` | — | — | — | todas |
| `E` | `MIL-CREDA**` | MIL-CREDA | — | — | todas |
| `F` | `MIL-CREDA*` | MIL-CREDA | sí | — | todas |
| `G` | `MIL-CREDA` | MIL-CREDA | sí | sí | todas |
| `SU` | `MIL-CREDA-U` | MIL-CREDA | sí | sí | 10, selección regular |
| `SA` | `MIL-CREDA-A` | MIL-CREDA | sí | sí | 10, selección arbitraria |
| `SK` | `MIL-CREDA-K` | MIL-CREDA | sí | sí | 10, las de mayor atención |

Los últimos tres mantienen fijo el presupuesto de instancias y se diferencian solo
en la regla que lo gasta, así que `MIL-CREDA-U → MIL-CREDA-K` y
`MIL-CREDA-A → MIL-CREDA-K` son atribuibles a la regla. `MIL-CREDA-K → MIL-CREDA`
es la pregunta aparte de cuánto cuesta el presupuesto en sí.

Todo lo de abajo está acotado por `config.py`, y solo dos constantes separan esta
corrida de la completa. **Leer el encabezado de cada tabla antes que sus números**:
por debajo del piso de repeticiones declarado no se otorga ningún veredicto y el
motivo queda estampado.

In [ ]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
import json

from IPython.display import Markdown, display

from MIL_CREDA_Benchmark import config, harness, tables


def show(text: str) -> None:
    """Una tabla se muestra como tabla, no como texto de ancho fijo.

    Es la misma cadena que va al registro, renderizada. Nada se vuelve a
    calcular acá: si esto y el archivo dijeran cosas distintas, habría dos
    versiones del mismo número y ninguna forma de saber cuál se movió.
    """
    display(Markdown(text))


# Los mismos registros que dejó la corrida, nunca recalculados: si esta
# celda y el archivo dijeran cosas distintas, habría dos versiones del mismo
# número y ninguna forma de saber cuál se movió.
runs = [json.loads(line) for line in
        (config.RESULTS / "runs.jsonl").read_text().splitlines() if line.strip()]
summary = json.loads((config.RESULTS / "summary.json").read_text())
reduction = harness.Reduction(**summary["reduction"])
registro_busqueda = harness.search_record()

print(f"{len(runs)} corridas · {len(summary['grid'])} transferencias")

## 0 · El techo de cada familia

La rejilla entera de la búsqueda, una fila por familia, con el techo elegido
marcado en su propia celda. Va la rejilla y no solo el ganador porque un techo que
gana entre cuatro puntajes idénticos y uno que gana por una diferencia real son el
mismo número y no la misma evidencia, y este escalar gobierna todas las tablas de
abajo.

In [ ]:
show(tables.objective("ceilings"))

In [ ]:
show(tables.render_ceilings(registro_busqueda, markdown=True))

In [ ]:
show(tables.conclusion_ceilings(registro_busqueda))

## Bajo qué corrió todo esto

Los límites de la corrida se dicen una vez, acá, y atan todo lo que sigue. No se
repiten en el encabezado de cada tabla: un aviso que aparece ocho veces enseña a
saltearlo, que es lo contrario de para lo que está.

In [ ]:
show(tables.stamp(summary["reduction"], markdown=True))

## 1 · Tiempo de entrenamiento

Los segundos que tarda una corrida completa de cada método en cada transferencia,
con la misma máquina, el mismo backbone y la misma cantidad de pasos. Va primero
porque un método que gana pagando diez veces el cómputo no gana lo mismo que uno que
gana gratis, y porque el costo es lo único que se compara limpio aunque las dos
familias predigan sobre unidades distintas. Buscamos el número más bajo, y sobre
todo que la diferencia entre métodos sea chica: un método más lento sigue siendo
utilizable, uno diez veces más lento deja de serlo.

In [ ]:
show(tables.objective("seconds"))

In [ ]:
show(tables.render(runs, "seconds", summary["reduction"], markdown=True))

In [ ]:
show(tables.conclusion(runs, "seconds", summary["reduction"]))

## 2 · Exactitud en el dominio fuente

La proporción de bolsas de evaluación del dominio **fuente** que cada método
clasifica bien: el dominio cuyas etiquetas sí vio durante el entrenamiento. Va antes
que la de destino porque es su complemento, y sin ella la tabla de destino no
distingue un éxito de una degeneración — un método que sube en destino rompiendo la
fuente aparece como ganador si solo se mira una tabla. Buscamos que sea alta, y
sobre todo que **no caiga** cuando se le suma la adaptación.

In [ ]:
show(tables.objective("sourceAccuracy"))

In [ ]:
show(tables.render(runs, "sourceAccuracy", summary["reduction"], markdown=True))

In [ ]:
show(tables.conclusion(runs, "sourceAccuracy", summary["reduction"]))

### 2b · Los peldaños en fuente

La diferencia entre los dos métodos de cada peldaño, transferencia por transferencia,
restada en el orden en que el peldaño se nombra: izquierda menos derecha. Un valor
**negativo** quiere decir que el de la derecha quedó por encima. La tabla anterior
dice quién está adelante; solo esta dice **qué componente** lo puso ahí, porque los
dos métodos de un peldaño se diferencian en una sola cosa. Buscamos peldaños que se
inclinen para el mismo lado en las seis transferencias: una media que promedia seis
acuerdos y una que promedia tres contra tres se ven idénticas y dicen cosas opuestas.

In [ ]:
show(tables.objective("rungs"))

In [ ]:
show(tables.render_rungs(summary, "sourceAccuracy", markdown=True))

In [ ]:
show(tables.conclusion_rungs(summary, "sourceAccuracy"))

## 3 · Exactitud en el dominio destino

La proporción de bolsas de evaluación del dominio **destino** —aquel cuyas etiquetas
el método nunca vio— que clasifica bien. Es la pregunta del problema: todo lo demás
en este cuaderno existe para poder leer esta tabla sin equivocarse. Buscamos que sea
alta, leída siempre junto a la de fuente y nunca sola, porque una subida acá pagada
con una caída allá no es adaptación sino un intercambio.

In [ ]:
show(tables.objective("targetAccuracy"))

In [ ]:
show(tables.render(runs, "targetAccuracy", summary["reduction"], markdown=True))

In [ ]:
show(tables.conclusion(runs, "targetAccuracy", summary["reduction"]))

### 3b · Los peldaños en destino

Lo mismo que 2b sobre el dominio destino, con la misma resta: izquierda menos
derecha, y un negativo es el de la derecha por encima. Acá es donde se lee qué aporta
cada componente al problema que se quiere resolver, así que buscamos lo mismo —
acuerdo entre las seis transferencias— con una lectura extra: un peldaño que se
inclina en fuente y no en destino es una diferencia que no llegó a donde importaba.

In [ ]:
show(tables.objective("rungs"))

In [ ]:
show(tables.render_rungs(summary, "targetAccuracy", markdown=True))

In [ ]:
show(tables.conclusion_rungs(summary, "targetAccuracy"))

In [ ]:
from MIL_CREDA_Benchmark import figures

## 4 · Las curvas del término supervisado

El término supervisado a lo largo del entrenamiento, un panel por transferencia,
para los métodos que lo comparten con un término de adaptación. Acá aparece un
término de adaptación que desestabiliza el ajuste: leer la curva de adaptación
sola llamaría bien portado a un término mientras la clasificación con la que
comparte objetivo se deshace por debajo. Cada curva es la mediana entre semillas
con banda intercuartil, nunca una corrida sola. Buscamos que el ajuste no se
degrade cuando se le suma la adaptación.

In [ ]:
show(tables.objective("supervised"))

In [ ]:
# Se muestra y se guarda en la misma llamada: la copia vectorial queda en Results
# y el cuaderno muestra la imagen, no su nombre de archivo.
display(figures.inline(figures.supervised_curves(config.RESULTS / "curves" / "supervised.pdf")))

## 5 · Las curvas del término de adaptación

Cuánto vale cada término de adaptación a lo largo del entrenamiento. La banda
sombreada es [0, 1]: la §5 normaliza los términos de MIL-CREDA exactamente sobre
ese intervalo y la puntuación del trabajo previo no tiene esa cota, así que si una
curva se queda adentro de la banda —y si ocupa la misma parte de ella de una
transferencia a otra— es la afirmación misma y no una ilustración de ella.
Buscamos curvas dentro de la banda y estables entre pares de dominios.

In [ ]:
show(tables.objective("adaptation"))

In [ ]:
# Se muestra y se guarda en la misma llamada: la copia vectorial queda en Results
# y el cuaderno muestra la imagen, no su nombre de archivo.
display(figures.inline(figures.adaptation_curves(config.RESULTS / "curves" / "adaptation.pdf")))

## 6 · Cuánto del objetivo comanda cada término

Qué proporción del objetivo se lleva de verdad cada término declarado. Sin este
panel, «el término no tuvo efecto» y «el término no tuvo peso» son la misma
figura. El coeficiente está fijo en `RAMP_CEILING` para todos los métodos, y fijar
el coeficiente no fija la proporción: un término cuya magnitud difiere en un orden
entre métodos es una diferencia que nadie declaró, y un peldaño que la ignora le
acredita al mecanismo lo que hizo la escala. Buscamos proporciones comparables
entre métodos, porque es lo que hace legible al peldaño.

In [ ]:
show(tables.objective("contribution"))

In [ ]:
# Se muestra y se guarda en la misma llamada: la copia vectorial queda en Results
# y el cuaderno muestra la imagen, no su nombre de archivo.
display(figures.inline(figures.contribution_curves(config.RESULTS / "curves" / "contribution.pdf")))

## 7 · El registro

Escribe las mismas tablas y conclusiones de arriba en `report.txt` y `report.md`, y
al lado `runs.jsonl` y `summary.json`, que son lo que lee la fase dos. Una sesión
posterior tiene que poder saber que esta corrida ocurrió, bajo qué reducción y contra
qué revisión, y nada de eso vive fuera del repositorio. Se genera junto con los
resultados y nunca se escribe a mano: un resumen escrito a mano es una segunda fuente
de verdad, se desactualiza en silencio y se le cree igual.

In [ ]:
bloques = [
    harness.header(reduction),
    tables.stamp(summary["reduction"]),
    tables.render_ceilings(registro_busqueda),
    tables.conclusion_ceilings(registro_busqueda),
    tables.render(runs, "seconds", summary["reduction"]),
    tables.conclusion(runs, "seconds", summary["reduction"]),
    tables.render(runs, "sourceAccuracy", summary["reduction"]),
    tables.conclusion(runs, "sourceAccuracy", summary["reduction"]),
    tables.render_rungs(summary, "sourceAccuracy"),
    tables.conclusion_rungs(summary, "sourceAccuracy"),
    tables.render(runs, "targetAccuracy", summary["reduction"]),
    tables.conclusion(runs, "targetAccuracy", summary["reduction"]),
    tables.render_rungs(summary, "targetAccuracy"),
    tables.conclusion_rungs(summary, "targetAccuracy"),
]
(config.RESULTS / "report.txt").write_text("\n\n".join(bloques), encoding="utf-8")

(config.RESULTS / "report.md").write_text("\n\n".join([
    f"# Fase uno — {config.REVISION}",
    tables.stamp(summary["reduction"], markdown=True),
    "## 0 · El techo de cada familia",
    tables.render_ceilings(registro_busqueda, markdown=True),
    tables.conclusion_ceilings(registro_busqueda),
    "## 1 · Tiempo de entrenamiento (más bajo es mejor)",
    tables.render(runs, "seconds", summary["reduction"], markdown=True),
    tables.conclusion(runs, "seconds", summary["reduction"]),
    "## 2 · Exactitud en fuente (más alto es mejor)",
    tables.render(runs, "sourceAccuracy", summary["reduction"], markdown=True),
    tables.conclusion(runs, "sourceAccuracy", summary["reduction"]),
    "### 2b · Peldaños en fuente",
    tables.render_rungs(summary, "sourceAccuracy", markdown=True),
    tables.conclusion_rungs(summary, "sourceAccuracy"),
    "## 3 · Exactitud en destino (más alto es mejor)",
    tables.render(runs, "targetAccuracy", summary["reduction"], markdown=True),
    tables.conclusion(runs, "targetAccuracy", summary["reduction"]),
    "### 3b · Peldaños en destino",
    tables.render_rungs(summary, "targetAccuracy", markdown=True),
    tables.conclusion_rungs(summary, "targetAccuracy"),
]), encoding="utf-8")

print("escritos:")
for path in sorted(config.RESULTS.iterdir()):
    print(" ", path.relative_to(config.REPOSITORY))

In [ ]:
# El sello: contra qué código corrió este informe. Sin él, un informe viejo y uno
# recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())